In [89]:
import pandas as pd
import numpy as np
import pdfplumber
import os
import streamlit

In [68]:
stock_df = pd.read_csv("stock_ceaiuri.csv")
stock_df.head(5)

,Produs,Stoc,Prag_minim
0,Green apple,5,2
1,Chai Matcha,5,2
2,Nana Mint,5,2
3,Green Energy,5,2
4,Mango Passionfruit,5,2


In [69]:
def stock_status(row):
    if row['Stoc'] <= 0:
        return "CRITIC"
    elif row['Stoc'] <= row['Prag_minim']:
        return "LOW"
    else:
        return "OK"

In [70]:
def color_status(val):
    if val == "CRITIC":
        return "background-color: #ff4d4d"   # roșu
    elif val == "LOW":
        return "background-color: #ffd24d"   # galben
    elif val == "OK":
        return "background-color: #85e085"   # verde

In [71]:
def reset_stock(row , value = 5):
    row['Stoc'] = value
    return row["Stoc"]

In [72]:
def extract_tea_sales(pdf_path):
    all_lines = []
    with pdfplumber.open(pdf_path) as pdf:
        
        for page in pdf.pages:
            text = page.extract_text()
            lines = text.split("\n")
            all_lines.extend(lines)

    tea_sales = []
    for line in all_lines:
        if '100 G Buc' in line:
            parts = line.split()
            
         # găsim poziția unde apare '100'
            idx_100 = parts.index('100')
         # numele produsului = cuvintele dintre 1 și '100'
            name = ' '.join(parts[1:idx_100])
            
        # cantitatea e după 'Buc'
            qty = parts[parts.index("Buc")+1]
            qty=int(float(qty))
            
            tea_sales.append((name,qty))
    sales_df = pd.DataFrame(tea_sales, columns=["Produs", "Vandut"])
    return sales_df

In [73]:
def load_all_sales(report_folder):
    all_sales = []
    for file in os.listdir(report_folder):
        if file.lower().endswith('.pdf'):
            path = '../reports/' + file
            date = file.split(" ")[-1].replace('.pdf',"")
            
            sales = extract_tea_sales(path)
            sales['Data'] = date
            
            all_sales.append(sales)
    all_sales = pd.concat(all_sales, ignore_index=True)
    return all_sales

In [74]:
all_sales = load_all_sales('../reports')

In [75]:
def aggregate_sales(all_sales):
    
    all_sales = (all_sales.groupby('Produs')['Vandut'].sum().reset_index())
    all_sales['Produs'] = all_sales['Produs'].str.lower()
    return all_sales

In [76]:
def update_inventory(stock_df,all_sales):
    stock_df['Produs'] = stock_df['Produs'].str.lower()
    merged = stock_df.merge(all_sales,on = 'Produs' , how = "left")
    merged['Vandut'] = merged['Vandut'].fillna(0).astype(int)
    merged['Stoc'] = merged['Stoc'] - merged['Vandut']
    merged['Status'] = merged.apply(stock_status, axis = 1)
    return merged

In [77]:
#update_inventory(stock_df=stock_df,all_sales=all_sales)

In [86]:
def run_inventory_pipeline(stock_df,report_folder):
    all_sales = load_all_sales(report_folder)
    sales_total = aggregate_sales(all_sales)
    updated_stock = update_inventory(stock_df , sales_total)
    updated_stock['Status'] = updated_stock.apply(stock_status,axis=1)
    return updated_stock

In [87]:
final_report = run_inventory_pipeline(stock_df,'../reports')

In [88]:
final_report

,Produs,Stoc,Prag_minim,Vandut,Status
0,green apple,5,2,0,OK
1,chai matcha,5,2,0,OK
2,nana mint,5,2,0,OK
3,green energy,5,2,0,OK
4,mango passionfruit,5,2,0,OK
5,spicy inspiration,5,2,0,OK
6,ginger orange,4,2,1,OK
7,jasmins herb basket,5,2,0,OK
8,a breeze of lavender,5,2,0,OK
9,boost & energy,5,2,0,OK


In [91]:
%%writefile dashboard.py

import streamlit as st

st.title("Tea Inventory Dashboard")
st.write("Welcome to the tea inventory system")

Writing dashboard.py


In [92]:
final_report.to_csv("inventory_report.csv", index=False)

In [93]:
!jupyter nbconvert --to script Tea_Stock_Project.ipynb

[NbConvertApp] Converting notebook Tea_Stock_Project.ipynb to script
[NbConvertApp] Writing 3500 bytes to Tea_Stock_Project.py
